# 第5部 2台をつなぐ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asg1223/rcj-soccer-ml/blob/main/notebooks/05_two_boards.ipynb)

このノートブックは、教材のページのコードを順番に並べたものです（`tools/build_notebooks.py` で自動生成）。
上のセルから順に `Shift + Enter` で実行してください。

In [ ]:
# Colab のときだけ、教材リポジトリを取ってきて読み込めるようにする
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q --depth 1 https://github.com/Asg1223/rcj-soccer-ml.git /content/rcj-soccer-ml
    sys.path.insert(0, "/content/rcj-soccer-ml")
print("Colab で実行中" if IN_COLAB else "ローカルで実行中")

## 5-2 パケットを設計する

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part5-two-boards/packet/

### 手を動かす 1：Python でパケットを作る

In [ ]:
from rcjml.packet import encode, Parser, pack_ir, unpack_ir, pack_ball, unpack_ball, crc8, T_IR_RAW, T_BALL

values = [650, 420, 80, 0, 0, 0, 0, 0, 0, 0, 51, 430]
pkt = encode(T_IR_RAW, seq=7, payload=pack_ir(values))
print(len(pkt), "バイト:", pkt.hex(" "))

### 手を動かす 2：受け取る側（パーサ）

In [ ]:
parser = Parser()
print(parser.feed(pkt))                 # 1 つそろった
print(unpack_ir(parser.feed(pkt)[0].payload))

### 手を動かす 3：わざと壊して確かめる

In [ ]:
import random
rnd = random.Random(0)
stream = bytearray()
expected = 0
for k in range(200):
    vals = [rnd.randint(0, 1023) for _ in range(12)]
    p = bytearray(encode(T_IR_RAW, k, pack_ir(vals)))
    if k % 10 == 3:
        p[7] ^= 0x40                    # 10 回に 1 回、中身を 1 ビット壊す
    else:
        expected += 1
    stream += bytes(rnd.randint(0, 255) for _ in range(rnd.randint(0, 3)))   # ゴミのバイト
    stream += p

parser = Parser()
got = parser.feed(bytes(stream[5:]))    # 途中から受け取りはじめる
print(f"正しく受け取れた {parser.ok}（壊していないのは {expected}）, CRC で見つけた {parser.crc_errors}")

## 5-3 CoreS3 に表示する

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part5-two-boards/display/

### 手を動かす 1：スケッチを用意する

In [ ]:
import numpy as np
import keras
from rcjml import load_config
from rcjml.data import load_synthetic, split_by_session
from rcjml.nn import features, targets, evaluate, keras_loss, build_ballnet

cfg = load_config()
df = load_synthetic("ring12_large_normal")
train_all, test = split_by_session(df)
last = train_all.session.max()
train, val = train_all[train_all.session < last], train_all[train_all.session == last]
keras.utils.set_random_seed(0)
model = build_ballnet()
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras_loss())
model.fit(features(train, cfg), targets(train), validation_data=(features(val, cfg), targets(val)),
          epochs=400, batch_size=64, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)])
print(evaluate(model.predict(features(test, cfg), verbose=0), test))

In [ ]:
from rcjml.config import PACKAGE_ROOT
from rcjml.export import export_to_sketch

sketch = PACKAGE_ROOT / "firmware" / "cores3" / "ball_display"
for f in export_to_sketch(model, sketch, allow_public=True):
    print(f.name)

## 5-5 制御ループに組み込む：推定でボールを追いかける

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part5-two-boards/control/

### 手を動かす 1：シミュレータで動きを見る

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rcjml.sim.field import (chase_rule, success_rate, run_episode, MAX_SPEED,
                             FIELD_W, FIELD_H, GOAL_W)

def chase(obs):
    return chase_rule(obs["angle_deg"], obs["distance_cm"], obs["holding"], obs["goal_angle_deg"])

def straight(obs):
    if obs["holding"]:
        return obs["goal_angle_deg"], MAX_SPEED
    return obs["angle_deg"], MAX_SPEED

for name, policy in [("straight", straight), ("chase (go around)", chase)]:
    r = success_rate(policy, n=300)
    print(f"{name:18s} ゴール {r['goal_rate']:.0%}  外に出した {r['out_rate']:.0%}  "
          f"ゴールまで {r['mean_steps_to_goal'] * 0.05:.2f} 秒")

In [ ]:
def draw(ep, ax, title):
    path = np.array([p[0] for p in ep.path])
    balls = np.array([p[1] for p in ep.path])
    ax.plot(path[:, 0], path[:, 1], color="tab:blue", label="robot")
    ax.plot(balls[:, 0], balls[:, 1], color="tab:orange", label="ball")
    ax.scatter(*path[0], color="tab:blue")
    ax.scatter(*balls[0], color="tab:orange")
    ax.plot([-GOAL_W / 2, GOAL_W / 2], [FIELD_H / 2] * 2, color="gold", lw=6)
    ax.set_xlim(-FIELD_W / 2, FIELD_W / 2)
    ax.set_ylim(-FIELD_H / 2, FIELD_H / 2 + 5)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(8, 5))
for ax, (name, policy) in zip(axes, [("straight", straight), ("chase", chase)]):
    ep = run_episode(policy, np.random.default_rng(3))
    draw(ep, ax, f"{name}: {'GOAL' if ep.goal else 'out' if ep.out else 'time up'}")
axes[0].legend(fontsize=8)
plt.show()

### 手を動かす 2：推定の良さで、動きはどう変わるか

In [ ]:
from rcjml.sim import simulate, sample_session, baseline_angle
from rcjml.sim.field import rel_ball, goal_angle
from rcjml.export import layers_from_keras, forward
from rcjml.nn import decode

layers = layers_from_keras(model)

def make_observe(kind, difficulty="normal", seed=1):
    rng = np.random.default_rng(seed)
    session = sample_session(difficulty, rng)
    def observe(ep):
        a, d = rel_ball(ep.robot, ep.ball)
        v = simulate(cfg, a, d, difficulty=difficulty, session=session, rng=rng)
        if kind == "vector sum":
            est = {"angle_deg": float(baseline_angle(cfg, v)[0]), "distance_cm": 40.0, "holding": ep.holding}
        else:
            r = decode(forward(layers, v / cfg.ir.max_reading))
            est = {"angle_deg": float(r["angle_deg"][0]), "distance_cm": float(r["distance_cm"][0]),
                   "holding": bool(r["p_hold"][0] > 0.5)}
        est["goal_angle_deg"] = goal_angle(ep.robot)      # ゴールの方向はわかるとする（第6部でカメラから）
        return est
    return observe

for name, obs in [("perfect", None), ("vector sum", make_observe("vector sum")), ("BallNet", make_observe("BallNet"))]:
    r = success_rate(chase, n=300, observe=obs)
    print(f"{name:11s} ゴール {r['goal_rate']:.0%}  ゴールまで {r['mean_steps_to_goal'] * 0.05:.2f} 秒")